In [44]:
import pandas as pd
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import re
import torch

df = pd.read_csv(
    "gutenberg_books.csv",
    dtype={"book_id": str}
)

print("Quantidade de livros:", len(df))
print("\nColunas disponíveis:")
print(df.columns.tolist())

df[["book_id", "title", "summary", "trecho"]].head(3)

sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

Quantidade de livros: 312

Colunas disponíveis:
['book_id', 'title', 'author', 'language', 'loc_class', 'subjects', 'release_date', 'ebook_url', 'summary', 'work_key', 'trecho', 'summary_normalized', 'summary_clean', 'tokens', 'tokens_no_punctuation', 'tokens_final', 'text_final', 'tokens_lemmatized', 'text_lemmatized', 'tokens_stemmed', 'text_stemmed']


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [45]:
df_sentimentos = df[
    ["book_id", "title", "trecho"]
].copy()

df_sentimentos["trecho"] = (
    df_sentimentos["trecho"]
    .fillna("")
    .astype(str)
    .str.strip()
)

validos = df_sentimentos["trecho"].ne("")

resultados = sentiment_model(
    df_sentimentos.loc[validos, "trecho"].tolist(),
    batch_size=16,
    truncation=True,
    max_length=512
)

df_sentimentos["sentimento"] = None
df_sentimentos["confianca"] = None

df_sentimentos.loc[validos, "sentimento"] = [
    resultado["label"] for resultado in resultados
]

df_sentimentos.loc[validos, "confianca"] = [
    resultado["score"] for resultado in resultados
]

df_sentimentos.head(10)

print("\nExemplos de classificação:")

display(
    df_sentimentos[
        ["title", "sentimento", "confianca", "trecho"]
    ].head(10)
)

Distribuição dos sentimentos:
sentimento
POSITIVE    10
NEGATIVE     5
Name: count, dtype: int64

Exemplos de classificação:


,title,sentimento,confianca,trecho
0,Pride and Prejudice,POSITIVE,0.999566,"Mrs. Gardiner had seen Pemberley, and known th..."
1,"Moby Dick; Or, The Whale",POSITIVE,0.914321,The acute policy dictating these movements was...
2,Crime and Punishment,NEGATIVE,0.992011,These exclamations and remarks checked Raskoln...
3,"Frankenstein; or, the modern prometheus",POSITIVE,0.991217,“Such were the events that preyed on the heart...
4,Alice's Adventures in Wonderland,NEGATIVE,0.934327,“What _can_ all that green stuff be?” said Ali...
5,Dracula,POSITIVE,0.537364,The poor dear was evidently terrified at somet...
6,The strange case of Dr. Jekyll and Mr. Hyde,POSITIVE,0.963975,"There was no mirror, at that date, in my room;..."
7,A Room with a View,POSITIVE,0.988348,"“Well, it isn’t everyone who could boast such ..."
8,Middlemarch,NEGATIVE,0.805027,"“Your mind is quite settled, then, Mary?” said..."
9,The Secret of Chimneys,POSITIVE,0.891418,"“There’s nothing much to explain,” said Anthon..."


In [46]:
distribuicao = (
    df_sentimentos["sentimento"]
    .dropna()
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Distribuição percentual:")
print(distribuicao)

Distribuição percentual:
sentimento
POSITIVE    66.67
NEGATIVE    33.33
Name: proportion, dtype: float64


In [47]:
ner_model = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"
)

df_ner = df[
    ["book_id", "title", "trecho"]
].copy()


df_ner["trecho"] = (
    df_ner["trecho"]
    .fillna("")
    .astype(str)
    .str.strip()
)

validos_ner = df_ner["trecho"].ne("")

amostra = df_ner.loc[validos_ner].head(10).copy()

amostra["entidades"] = [
    ner_model(texto)
    for texto in amostra["trecho"]
]

display(
    amostra[
        ["title", "entidades"]
    ]
)

linhas_entidades = []

for _, row in amostra.iterrows():
    for entidade in row["entidades"]:
        linhas_entidades.append({
            "book_id": row["book_id"],
            "title": row["title"],
            "entidade": entidade["word"],
            "tipo": entidade["entity_group"],
            "confianca": entidade["score"],
            "inicio": entidade["start"],
            "fim": entidade["end"]
        })

df_entidades = pd.DataFrame(linhas_entidades)

display(df_entidades.head(20))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,title,entidades
0,Pride and Prejudice,"[{'entity_group': 'PER', 'score': 0.9973851, '..."
1,"Moby Dick; Or, The Whale","[{'entity_group': 'PER', 'score': 0.9604601, '..."
2,Crime and Punishment,"[{'entity_group': 'PER', 'score': 0.99947387, ..."
3,"Frankenstein; or, the modern prometheus","[{'entity_group': 'PER', 'score': 0.99903464, ..."
4,Alice's Adventures in Wonderland,"[{'entity_group': 'PER', 'score': 0.7126694, '..."
5,Dracula,"[{'entity_group': 'ORG', 'score': 0.4792658, '..."
6,The strange case of Dr. Jekyll and Mr. Hyde,"[{'entity_group': 'PER', 'score': 0.9948364, '..."
7,A Room with a View,"[{'entity_group': 'LOC', 'score': 0.9872169, '..."
8,Middlemarch,"[{'entity_group': 'PER', 'score': 0.9181548, '..."
9,The Secret of Chimneys,"[{'entity_group': 'PER', 'score': 0.99841845, ..."


,book_id,title,entidade,tipo,confianca,inicio,fim
0,1342,Pride and Prejudice,Gardiner,PER,0.997385,5,13
1,1342,Pride and Prejudice,P,PER,0.994982,23,24
2,1342,Pride and Prejudice,##em,PER,0.944491,24,26
3,1342,Pride and Prejudice,Darcy,PER,0.856716,57,62
4,1342,Pride and Prejudice,P,PER,0.920670,188,189
5,1342,Pride and Prejudice,##em,PER,0.742504,189,191
6,1342,Pride and Prejudice,W,PER,0.996156,232,233
7,1342,Pride and Prejudice,##ick,PER,0.735296,233,236
8,1342,Pride and Prejudice,Mr,PER,0.582038,414,416
9,1342,Pride and Prejudice,Darcy,PER,0.797947,418,423


In [48]:
print("Total de entidades encontradas:", len(df_entidades))

print("\nQuantidade por tipo:")
display(
    df_entidades["tipo"].value_counts().to_frame("quantidade")
)

print("\nEntidades mais frequentes:")
display(
    df_entidades["entidade"]
    .value_counts()
    .head(15)
    .to_frame("frequencia")
)

Total de entidades encontradas: 65

Quantidade por tipo:


,quantidade
tipo,
PER,53
LOC,7
MISC,4
ORG,1



Entidades mais frequentes:


,frequencia
entidade,
Victor,3
##em,2
P,2
##skolniko,2
Ra,2
##red,2
Darcy,2
Sa,2
Mary,2
